In [1]:
from groq import Groq
from dotenv import load_dotenv
import os
import json

load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
print("Groq подключён!")

Groq подключён!


In [2]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print(response.choices[0].message.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [3]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an appointment scheduling assistant for a healthcare facility."},
        {"role": "user", "content": "I need to see a doctor next week."}
    ]
)
print(response.choices[0].message.content)

I'd be happy to help you schedule an appointment with a doctor. Can you please tell me what type of doctor you need to see? For example, are you looking for a primary care physician, a specialist, or perhaps an urgent care visit?

Also, what are your availability preferences for next week? Are you looking for a morning, afternoon, or evening appointment? And do you have any specific days that work better for you?


In [4]:
import json

# Описание функции для модели
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_availability",
            "description": "Check available appointment slots for a given date",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string",
                        "description": "Date in YYYY-MM-DD format"
                    }
                },
                "required": ["date"]
            }
        }
    }
]

print("Функция описана!")

Функция описана!


In [5]:
def check_availability(date):
    """Проверяет доступные слоты на дату"""
    
    # Пока это фейковые данные — потом подключим базу данных
    fake_schedule = {
        "2026-04-28": ["09:00", "11:00", "14:00"],
        "2026-04-29": ["10:00", "15:00"],
        "2026-04-30": []
    }
    
    slots = fake_schedule.get(date, [])
    
    if slots:
        return f"Available slots on {date}: {', '.join(slots)}"
    else:
        return f"No available slots on {date}"

# Тестируем
print(check_availability("2026-04-28"))
print(check_availability("2026-04-30"))
print(check_availability("2026-05-01"))

Available slots on 2026-04-28: 09:00, 11:00, 14:00
No available slots on 2026-04-30
No available slots on 2026-05-01


In [6]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an appointment scheduling assistant."},
        {"role": "user", "content": "Are there any slots on April 28, 2026?"}
    ],
    tools=tools
)

message = response.choices[0].message
print("Модель решила:", message.tool_calls)

Модель решила: [ChatCompletionMessageToolCall(id='zxst9vc8a', function=Function(arguments='{"date":"2026-04-28"}', name='check_availability'), type='function')]


In [7]:
# 1. Достаём что модель попросила
tool_call = message.tool_calls[0]
function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

print(f"Модель хочет вызвать: {function_name}")
print(f"С аргументами: {arguments}")

# 2. Вызываем функцию
result = check_availability(arguments["date"])
print(f"Результат: {result}")

# 3. Отправляем результат обратно модели
final_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an appointment scheduling assistant."},
        {"role": "user", "content": "Are there any slots on April 28, 2026?"},
        message,
        {"role": "tool", "content": result, "tool_call_id": tool_call.id}
    ],
    tools=tools
)

print("\nОтвет пациенту:")
print(final_response.choices[0].message.content)

Модель хочет вызвать: check_availability
С аргументами: {'date': '2026-04-28'}
Результат: Available slots on 2026-04-28: 09:00, 11:00, 14:00

Ответ пациенту:
There are available slots on April 28, 2026, at 09:00, 11:00, and 14:00.


In [8]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an appointment scheduling assistant."},
        {"role": "user", "content": "Do you have anything available on May 5, 2026?"}
    ],
    tools=tools
)

message = response.choices[0].message

if message.tool_calls:
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    result = check_availability(arguments["date"])
    
    final_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an appointment scheduling assistant."},
            {"role": "user", "content": "Do you have anything available on May 5, 2026?"},
            message,
            {"role": "tool", "content": result, "tool_call_id": tool_call.id}
        ],
        tools=tools
    )
    print("Ответ пациенту:")
    print(final_response.choices[0].message.content)
else:
    print("Модель ответила без функции:")
    print(message.content)

Ответ пациенту:
None


In [9]:
print("Content:", final_response.choices[0].message.content)
print("Tool calls:", final_response.choices[0].message.tool_calls)

Content: None
Tool calls: [ChatCompletionMessageToolCall(id='wvw6j55zj', function=Function(arguments='{"date":"2026-05-06"}', name='check_availability'), type='function')]


In [10]:
system_prompt = """You are an appointment scheduling assistant.
When a function returns no available slots, tell the patient 
honestly and ask if they want to check a different date.
Do NOT automatically check other dates yourself."""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Do you have anything available on May 5, 2026?"}
    ],
    tools=tools
)

message = response.choices[0].message
tool_call = message.tool_calls[0]
arguments = json.loads(tool_call.function.arguments)
result = check_availability(arguments["date"])

final_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Do you have anything available on May 5, 2026?"},
        message,
        {"role": "tool", "content": result, "tool_call_id": tool_call.id}
    ],
    tools=tools
)

print("Ответ пациенту:")
print(final_response.choices[0].message.content)

Ответ пациенту:
I've checked the schedule for May 5, 2026, and unfortunately, there are no available appointment slots on that date. Would you like to check a different date?


In [11]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "What should I take for a headache?"}
    ],
    tools=tools
)

message = response.choices[0].message

if message.tool_calls:
    print("Модель вызвала функцию (плохо!):")
    print(message.tool_calls)
else:
    print("Ответ пациенту:")
    print(message.content)


Ответ пациенту:
I'm not able to provide medical advice. If you're experiencing a headache, I recommend speaking with a healthcare professional or pharmacist for guidance on appropriate treatment options.


In [13]:
import sqlite3

conn = sqlite3.connect("scheduler.db")
cursor = conn.cursor()

# Таблица врачей
cursor.execute("""
    CREATE TABLE IF NOT EXISTS doctors (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        department TEXT NOT NULL
    )
""")

conn.commit()
print("Таблица doctors создана!")

Таблица doctors создана!


In [14]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS slots (
        id INTEGER PRIMARY KEY,
        doctor_id INTEGER NOT NULL,
        date TEXT NOT NULL,
        time TEXT NOT NULL,
        patient_name TEXT DEFAULT NULL,
        patient_phone TEXT DEFAULT NULL,
        insurance TEXT DEFAULT NULL,
        appointment_type TEXT DEFAULT 'primary',
        status TEXT DEFAULT 'available',
        FOREIGN KEY (doctor_id) REFERENCES doctors (id)
    )
""")

conn.commit()
print("Таблица slots создана!")

Таблица slots создана!


In [15]:
cursor.execute("""
    INSERT INTO doctors (name, department) VALUES
    ('Dr. Smith', 'Family Medicine'),
    ('Dr. Johnson', 'Family Medicine'),
    ('Dr. Patel', 'Internal Medicine'),
    ('Dr. Lee', 'Cardiology'),
    ('Dr. Garcia', 'Dermatology'),
    ('Dr. Williams', 'Orthopedics'),
    ('Dr. Chen', 'Pediatrics'),
    ('Dr. Brown', 'Obstetrics & Gynecology'),
    ('Dr. Davis', 'Psychiatry'),
    ('Dr. Wilson', 'Neurology'),
    ('Dr. Martinez', 'Ophthalmology'),
    ('Dr. Anderson', 'ENT'),
    ('Dr. Taylor', 'Urology'),
    ('Dr. Thomas', 'Gastroenterology'),
    ('Dr. Moore', 'Pulmonology'),
    ('Dr. Jackson', 'Endocrinology'),
    ('Dr. White', 'Rheumatology'),
    ('Dr. Harris', 'Allergy & Immunology')
""")

conn.commit()
print("Врачи добавлены!\n")

cursor.execute("SELECT * FROM doctors")
for row in cursor.fetchall():
    print(f"ID: {row[0]} | {row[1]:20s} | {row[2]}")

Врачи добавлены!

ID: 1 | Dr. Smith            | Family Medicine
ID: 2 | Dr. Johnson          | Family Medicine
ID: 3 | Dr. Patel            | Internal Medicine
ID: 4 | Dr. Lee              | Cardiology
ID: 5 | Dr. Garcia           | Dermatology
ID: 6 | Dr. Williams         | Orthopedics
ID: 7 | Dr. Chen             | Pediatrics
ID: 8 | Dr. Brown            | Obstetrics & Gynecology
ID: 9 | Dr. Davis            | Psychiatry
ID: 10 | Dr. Wilson           | Neurology
ID: 11 | Dr. Martinez         | Ophthalmology
ID: 12 | Dr. Anderson         | ENT
ID: 13 | Dr. Taylor           | Urology
ID: 14 | Dr. Thomas           | Gastroenterology
ID: 15 | Dr. Moore            | Pulmonology
ID: 16 | Dr. Jackson          | Endocrinology
ID: 17 | Dr. White            | Rheumatology
ID: 18 | Dr. Harris           | Allergy & Immunology


In [16]:
from datetime import datetime, timedelta

# Неделя с 28 апреля по 2 мая 2026
start_date = datetime(2026, 4, 28)
days = 5  # Пн-Пт

# Временные окна для каждого врача
morning_slots = ["09:00", "09:30", "10:00", "10:30", "11:00", "11:30"]
afternoon_slots = ["13:00", "13:30", "14:00", "14:30", "15:00", "15:30"]

# Получаем всех врачей
cursor.execute("SELECT id, name FROM doctors")
doctors = cursor.fetchall()

count = 0
for day in range(days):
    date = (start_date + timedelta(days=day)).strftime("%Y-%m-%d")
    for doctor_id, doctor_name in doctors:
        # Каждый врач работает утро ИЛИ день (чередуем)
        if doctor_id % 2 == 0:
            slots = morning_slots
        else:
            slots = afternoon_slots
        
        for time in slots:
            cursor.execute(
                "INSERT INTO slots (doctor_id, date, time) VALUES (?, ?, ?)",
                (doctor_id, date, time)
            )
            count += 1

conn.commit()
print(f"Создано {count} слотов!")

# Проверяем: слоты Dr. Smith на понедельник
print("\nDr. Smith, 28 апреля:")
cursor.execute("""
    SELECT slots.time, slots.status 
    FROM slots 
    JOIN doctors ON slots.doctor_id = doctors.id
    WHERE doctors.name = 'Dr. Smith' AND slots.date = '2026-04-28'
""")
for row in cursor.fetchall():
    print(f"  {row[0]} — {row[1]}")

Создано 540 слотов!

Dr. Smith, 28 апреля:
  13:00 — available
  13:30 — available
  14:00 — available
  14:30 — available
  15:00 — available
  15:30 — available


In [17]:
# Несколько слотов уже забронированы
bookings = [
    ("Anna Kowalska", "555-0101", "Blue Cross", 1, "2026-04-28", "13:00"),
    ("John Miller", "555-0102", "Aetna", 1, "2026-04-28", "14:00"),
    ("Maria Santos", "555-0103", None, 4, "2026-04-28", "09:00"),
    ("James Wilson", "555-0104", "United Health", 7, "2026-04-29", "13:00"),
    ("Sarah Park", "555-0105", "Cigna", 9, "2026-04-29", "13:30"),
]

for name, phone, insurance, doctor_id, date, time in bookings:
    cursor.execute("""
        UPDATE slots 
        SET patient_name = ?, patient_phone = ?, 
            insurance = ?, status = 'booked'
        WHERE doctor_id = ? AND date = ? AND time = ?
    """, (name, phone, insurance, doctor_id, date, time))

conn.commit()
print("Пациенты записаны!\n")

# Проверяем Dr. Smith 28 апреля
print("Dr. Smith, 28 апреля:")
cursor.execute("""
    SELECT slots.time, slots.status, slots.patient_name
    FROM slots
    JOIN doctors ON slots.doctor_id = doctors.id
    WHERE doctors.name = 'Dr. Smith' AND slots.date = '2026-04-28'
""")
for row in cursor.fetchall():
    name = row[2] if row[2] else "(свободно)"
    print(f"  {row[0]} — {row[1]:10s} | {name}")

Пациенты записаны!

Dr. Smith, 28 апреля:
  13:00 — booked     | Anna Kowalska
  13:30 — available  | (свободно)
  14:00 — booked     | John Miller
  14:30 — available  | (свободно)
  15:00 — available  | (свободно)
  15:30 — available  | (свободно)


In [18]:
def check_availability(date, doctor_name=None, department=None):
    """Проверяет свободные слоты в базе данных"""
    
    if doctor_name:
        cursor.execute("""
            SELECT doctors.name, doctors.department, slots.time
            FROM slots
            JOIN doctors ON slots.doctor_id = doctors.id
            WHERE slots.date = ? AND doctors.name = ? AND slots.status = 'available'
            ORDER BY slots.time
        """, (date, doctor_name))
    elif department:
        cursor.execute("""
            SELECT doctors.name, doctors.department, slots.time
            FROM slots
            JOIN doctors ON slots.doctor_id = doctors.id
            WHERE slots.date = ? AND doctors.department = ? AND slots.status = 'available'
            ORDER BY doctors.name, slots.time
        """, (date, department))
    else:
        cursor.execute("""
            SELECT doctors.name, doctors.department, slots.time
            FROM slots
            JOIN doctors ON slots.doctor_id = doctors.id
            WHERE slots.date = ? AND slots.status = 'available'
            ORDER BY doctors.department, doctors.name, slots.time
        """, (date,))
    
    results = cursor.fetchall()
    
    if not results:
        return f"No available slots on {date}"
    
    output = f"Available slots on {date}:\n"
    for doctor, dept, time in results:
        output += f"  {time} — {doctor} ({dept})\n"
    return output

# Тестируем!
print(check_availability("2026-04-28", doctor_name="Dr. Smith"))

Available slots on 2026-04-28:
  13:30 — Dr. Smith (Family Medicine)
  14:30 — Dr. Smith (Family Medicine)
  15:00 — Dr. Smith (Family Medicine)
  15:30 — Dr. Smith (Family Medicine)



In [19]:
# 1. Конкретный врач
print("=== Dr. Smith ===")
print(check_availability("2026-04-28", doctor_name="Dr. Smith"))

# 2. Целое отделение
print("=== Cardiology ===")
print(check_availability("2026-04-28", department="Cardiology"))

# 3. Вся клиника
print("=== Все врачи ===")
print(check_availability("2026-04-28"))

=== Dr. Smith ===
Available slots on 2026-04-28:
  13:30 — Dr. Smith (Family Medicine)
  14:30 — Dr. Smith (Family Medicine)
  15:00 — Dr. Smith (Family Medicine)
  15:30 — Dr. Smith (Family Medicine)

=== Cardiology ===
Available slots on 2026-04-28:
  09:30 — Dr. Lee (Cardiology)
  10:00 — Dr. Lee (Cardiology)
  10:30 — Dr. Lee (Cardiology)
  11:00 — Dr. Lee (Cardiology)
  11:30 — Dr. Lee (Cardiology)

=== Все врачи ===
Available slots on 2026-04-28:
  09:00 — Dr. Harris (Allergy & Immunology)
  09:30 — Dr. Harris (Allergy & Immunology)
  10:00 — Dr. Harris (Allergy & Immunology)
  10:30 — Dr. Harris (Allergy & Immunology)
  11:00 — Dr. Harris (Allergy & Immunology)
  11:30 — Dr. Harris (Allergy & Immunology)
  09:30 — Dr. Lee (Cardiology)
  10:00 — Dr. Lee (Cardiology)
  10:30 — Dr. Lee (Cardiology)
  11:00 — Dr. Lee (Cardiology)
  11:30 — Dr. Lee (Cardiology)
  13:00 — Dr. Garcia (Dermatology)
  13:30 — Dr. Garcia (Dermatology)
  14:00 — Dr. Garcia (Dermatology)
  14:30 — Dr. Garci

In [20]:
def book_slot(date, time, doctor_name, patient_name, patient_phone, insurance=None):
    """Записывает пациента на приём"""
    
    # Проверяем: слот существует и свободен?
    cursor.execute("""
        SELECT slots.id, slots.status
        FROM slots
        JOIN doctors ON slots.doctor_id = doctors.id
        WHERE slots.date = ? AND slots.time = ? AND doctors.name = ?
    """, (date, time, doctor_name))
    
    result = cursor.fetchone()
    
    if not result:
        return f"No slot found for {doctor_name} on {date} at {time}"
    
    if result[1] == 'booked':
        return f"Sorry, {doctor_name} at {time} on {date} is already booked"
    
    # Записываем!
    cursor.execute("""
        UPDATE slots
        SET patient_name = ?, patient_phone = ?, 
            insurance = ?, status = 'booked'
        WHERE id = ?
    """, (patient_name, patient_phone, insurance, result[0]))
    
    conn.commit()
    return f"Appointment confirmed: {doctor_name} on {date} at {time} for {patient_name}"

# Тестируем!
print(book_slot("2026-04-28", "13:30", "Dr. Smith", "Test Patient", "555-9999", "Aetna"))

Appointment confirmed: Dr. Smith on 2026-04-28 at 13:30 for Test Patient


In [21]:
print("Dr. Smith, 28 апреля:")
cursor.execute("""
    SELECT slots.time, slots.status, slots.patient_name
    FROM slots
    JOIN doctors ON slots.doctor_id = doctors.id
    WHERE doctors.name = 'Dr. Smith' AND slots.date = '2026-04-28'
""")
for row in cursor.fetchall():
    name = row[2] if row[2] else "(свободно)"
    print(f"  {row[0]} — {row[1]:10s} | {name}")

Dr. Smith, 28 апреля:
  13:00 — booked     | Anna Kowalska
  13:30 — booked     | Test Patient
  14:00 — booked     | John Miller
  14:30 — available  | (свободно)
  15:00 — available  | (свободно)
  15:30 — available  | (свободно)


In [22]:
# Тест 1: занятый слот
print("Тест 1 — занятый слот:")
print(book_slot("2026-04-28", "13:00", "Dr. Smith", "Hacker", "000-0000"))

# Тест 2: несуществующий слот
print("\nТест 2 — несуществующий слот:")
print(book_slot("2026-12-25", "09:00", "Dr. Smith", "Santa", "000-0000"))

# Тест 3: нормальная запись
print("\nТест 3 — свободный слот:")
print(book_slot("2026-04-28", "15:00", "Dr. Smith", "Maria Lopez", "555-7777", "United Health"))

Тест 1 — занятый слот:
Sorry, Dr. Smith at 13:00 on 2026-04-28 is already booked

Тест 2 — несуществующий слот:
No slot found for Dr. Smith on 2026-12-25 at 09:00

Тест 3 — свободный слот:
Appointment confirmed: Dr. Smith on 2026-04-28 at 15:00 for Maria Lopez
